In [ ]:
# Import libraries
from delta import DeltaTable
from pyspark.sql.functions import col, when, first, max
import json

In [ ]:
# Define source schema and tables for fact_demand table in gold
source_schema = 'dbo'
source_tables = ['generation', 'generation_forecast']

In [ ]:
# Define parameters with None, as values will come from pipeline activities
# Source
source_lakehouse_path = None 

# Target
target_lakehouse_path = None
target_schema = None
target_table = None

# Bronze Control Table
warehouse_metadata_path = None 

In [ ]:
# Define variables using values from parameters
source_paths = [f'{source_lakehouse_path}/Tables/{source_schema}/{table}' for table in source_tables]
target_path = f'{target_lakehouse_path}/Tables/{target_schema}/{target_table}'

# Load Helper Notebooks

Load helper notebook with the schema, merge keys and indicator mapping definition of gold tables inside "gold_objects" dictionary: 


In [ ]:
%run ./NB_Gold_Table_Schemas

In [ ]:
# Use gold_objects dictionary to define aditional variables
target_table_schema = gold_objects[f"{target_table}"]["schema"]

# Read and Transform New Data

In [ ]:
# Create a list of dataframes with data of tables in source_paths list. Only read unique values of relevant columns 
dfs = [
    spark.read
        .format("delta")
        .load(path)
        .select(
            col("indicator_id"),
            col("indicator_name")
        )
        .dropDuplicates()
    for path in source_paths
]

In [ ]:
# Use the first dataframe in dfs as df, then iterate over the rest of the dataframes in the list performing an union between them
df = dfs[0]

for source_df in dfs[1:]:
    df = df.unionByName(source_df)

# Remove duplicates across all sources
df = df.dropDuplicates(["indicator_id", "indicator_name"])

In [ ]:
# Read bronze control table, where each row represents an indicator and there are columns with short name and display name needed for reporting
df_control = (
    spark.read
        .format("delta")
        .load(f"{warehouse_metadata_path}/Tables/config/bronze_control")
        .select(
            col("esios_indicator_id"),
            col("esios_indicator_short_name"),
            col("indicator_display_name")
        )
)

In [ ]:
# Join the dataframes to add short and display names in df, then use select so esios_indicator_id is not in the final df
df = (
    df
    .join(df_control, on = df.indicator_id==df_control.esios_indicator_id, how = 'left')
    .select(
        col("indicator_id"),
        col("indicator_name"),
        col("esios_indicator_short_name"),
        col("indicator_display_name")
    )
)

In [ ]:
# Classify indicators by resource type
renewables = [541, 542, 543, 546, 551, 1295, 1294]
non_renewables = [549, 550, 547, 548]

df = df.withColumn(
    "resource_type",
    when(col("indicator_id").isin(renewables), "Renewable")
    .when(col("indicator_id").isin(non_renewables), "Non-renewable")
    .otherwise("Undefined")
)

# Update Table

In [ ]:
# Full load
# Create the table if it doesnt exists
try:
    delta_table = DeltaTable.forPath(spark, target_path)
    print('Table already exists')
except:
    delta_table = DeltaTable.createIfNotExists(spark) \
    .location(target_path) \
    .addColumns(target_table_schema) \
    .execute()
    print('Table has been created')

# Recreate the table without infering schema
df.write.format("delta").mode("overwrite").save(target_path) 

# Statistics
rows_inserted = df.count()
rows_updated = 0

In [ ]:
# File a dictionary with operational statistics
result = {
    "maxdate": None,
    "rows_inserted": rows_inserted,
    "rows_updated": rows_updated
}

# Make the result values queriable for other activities in the pipeline
mssparkutils.notebook.exit(json.dumps(result))